# Preprocessing

## Imports and paths

In [1]:
import pandas as pd
import re
import pdfplumber # doesnt do OCR - we do not even want pdfs which are not machine readable (image-only pdf)
import unicodedata
from pathlib import Path
import os

# 1. setup paths
BASE_DIR = Path('..')
DATA_DIR = BASE_DIR / "data"
DEBUG_DIR = DATA_DIR / "debug_steps"
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

# 2. Loading checklist and selecting for exampole 5 samples
df = pd.read_csv(DATA_DIR / "99_eda" / "outputs" / "eda_table.csv")
target_labels = ["1_GOLD_MODERATION", "2_STRONG_CANDIDATE", "3_PENALTY_NO_MODERATION"]
df_filtered = df[df["eda_label"].isin(target_labels)].copy()
print(len(df_filtered))

# #########  Now get diversified documents  #########
# get one decision from NS SR
picked = []
ns = df_filtered[df_filtered["court"].eq("NS SR")]
if not ns.empty:
    picked.append(ns.sort_values("date", ascending=False).iloc[0])

# 2 - get 1 doc from different courts
others = df_filtered[~df_filtered["court"].eq("NS SR")]
one_per_court = (
    others.sort_values("date", ascending=False)
          .drop_duplicates(subset=["court"], keep="first")
)

# 3) fill it if there is no 5 different courts (now i have)
need = 5 - len(picked)
picked += [row for _, row in one_per_court.head(need).iterrows()]

df_selected = pd.DataFrame(picked)
SAMPLE_FILES = df_selected.to_dict("records")


# Storage for our processing pipeline (list of dicts) - semi results are stored here
pipeline_data = []

print(f" Selected {len(SAMPLE_FILES)} files for testing.")
print(f"Debugginh outputs will be  saved to: {DEBUG_DIR.resolve()}")

146
 Selected 5 files for testing.
Debugginh outputs will be  saved to: /Users/stefanec/STU_FIIT/bachelor-thesis-legal-text-information-extraction/data/debug_steps


In [2]:
print(df.columns)

Index(['filename', 'rel_path', 'case_id', 'court', 'date', 'eda_label',
       'est_tokens'],
      dtype='object')


## 3.1 Extracting text from PDFs 

- pipeline_data - we will add there edited versions of text
- we do not do OCR - using pdfplumber
- we keeps the structure of text - we save text into list raw_pages - index 0 = page 1, index 1 = page 2 (so we can later say - this proof is on second page) 

In [3]:
pipeline_data = [] #resetting pipeline

for row in SAMPLE_FILES:
    pdf_path = BASE_DIR / row['rel_path']
    doc_data = {
        "filename": row['filename'],
        "raw_pages": [],  # list of strings (one per page)
        "page_map": []    # list of page numbers
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages):
                text = page.extract_text() or ""    # extracting text from page
                doc_data["raw_pages"].append(text)
                doc_data["page_map"].append(i + 1)
                
        pipeline_data.append(doc_data)
        
        # Debug save
        debug_path = DEBUG_DIR / f"{row['filename']}_01_raw.txt"
        with open(debug_path, "w", encoding="utf-8") as f:
            f.write("\n--- PAGE BREAK ---\n".join(doc_data["raw_pages"]))
            
    except Exception as e:
        print(f"There is errror in reading {row['filename']}: {e}")

print(f"Extracted text from {len(pipeline_data)} files.")
print("Check folder 'debug_steps' for files ending in '_01_raw.txt'")

Extracted text from 5 files.
Check folder 'debug_steps' for files ending in '_01_raw.txt'


This cell has extracted text without damaged characters and also kept structure (heasders, pagination is there - we did not remove it yet)

## 3.2 Light unicode normalization 
- Cleaning invisible characters
- NFC normalization - ensures that letters like č,š,á  are saved as one character, not as  2 (c + ˇ)
- removing nonvisible characters like \u200b (zero-width space - allowed break point ) or \u00ad (soft hyphen - they ensures that words are divided on good position) - for example rozhodnutie would look like roz\u00adhodnutie
- also fixing new lines (\r\n -> \n)


In [4]:

# Removing zero-width spaces, fixing newlines (\r\n -> \n), NFC normalization.

def clean_light(text: str) -> str:
    if not text: return ""
    # 1. NFC (fix accents)
    text = unicodedata.normalize('NFC', text)
    # 2. Remove invisible trash
    text = text.replace('\u200b', '').replace('\u00ad', '')
    # 3. Unify newlines
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    return text



for doc in pipeline_data:
    # Apply to every page
    doc["clean_pages"] = [clean_light(p) for p in doc["raw_pages"]]
    
    # DEBUG SAVE
    debug_path = DEBUG_DIR / f"{doc['filename']}_02_light.txt"
    with open(debug_path, "w", encoding="utf-8") as f:
        f.write("\n--- PAGE BREAK ---\n".join(doc["clean_pages"]))


In [5]:
# lets review first document and first page
doc_test = pipeline_data[0] 

raw_snippet = doc_test["raw_pages"][0][:500]   # Prvých 500 znakov PRED úpravou
clean_snippet = doc_test["clean_pages"][0][:500] # Prvých 500 znakov PO úprave

print(f"file: {doc_test['filename']}")
print("-" * 50)

print("BEFORE (Raw - we can see hidden chars):")
# Funkcia repr() ukáže \r, \n, \t a unicode znaky explicitne
print(repr(raw_snippet)) 

print("\n" + "-" * 50 + "\n")

print("AFTER (Clean - fixed):")
print(repr(clean_snippet))

file: NS_SR_5Obdo_14_2023_00_dokument.pdf
--------------------------------------------------
BEFORE (Raw - we can see hidden chars):
'Súd: Najvyšší súd\nSpisová značka: 5Obdo/14/2023\nIdentifikačné číslo súdneho spisu: 7811201291\nDátum vydania rozhodnutia: 25. 04. 2024\nMeno a priezvisko sudcu, VSÚ: JUDr. Andrea Moravčíková\nECLI: ECLI:SK:NSSR:2024:7811201291.4\nROZSUDOK V MENE\nSLOVENSKEJ REPUBLIKY\nNajvyšší súd Slovenskej republiky v senáte zloženom z predsedníčky senátu\nJUDr. Andrey Moravčíkovej, PhD., a členiek senátu JUDr. Ivany Nemčekovej\na JUDr. Lenky Praženkovej, v spore žalobcu METALFIN, s. r. o., Mudroňova 36, Košice, IČO: '

--------------------------------------------------

AFTER (Clean - fixed):
'Súd: Najvyšší súd\nSpisová značka: 5Obdo/14/2023\nIdentifikačné číslo súdneho spisu: 7811201291\nDátum vydania rozhodnutia: 25. 04. 2024\nMeno a priezvisko sudcu, VSÚ: JUDr. Andrea Moravčíková\nECLI: ECLI:SK:NSSR:2024:7811201291.4\nROZSUDOK V MENE\nSLOVENSKEJ REPUBLIKY\nNajvyš

PDF files often contain invisible characters (like zero-width spaces) and Windows line endings (\r\n) that confuse Python, but these files doesnt seem to contain them. Of course i will keep it for robustness. If I would not  fix them, my Regex patterns in the next steps might fail.

I used the repr() command to see the "raw" text. I confirmed that all \r\n are changed to \n and hidden Unicode trash is gone.

## 3.3 Extraction of data in header
Extracting Court, Date, Case ID from the first page(s).

I extract key metadata from the document header. But the structure of legal documents varies a little bit, so I use REGEX to find specific patterns in first 2000 characters of decision

In [6]:

def extract_meta(header_text: str):
    meta = {}
    # Regexes based on heuristics
    # 1. Date (DD.MM.YYYY or DD. MM. YYYY) - also deleting spaeces, so we have format 04.06.2012
    date_match = re.search(r"(\d{1,2}\.\s*\d{1,2}\.\s*\d{4})", header_text)
    if date_match: meta['date_raw'] = date_match.group(1).replace(" ", "")
    
    # 2. Case ID (Spisová značka)
    # Looks for pattern like "3Cob/224/2014"  after "Spisová značka:"
    sz_match = re.search(r"Spisová\s*značka:?\s*([\w\d/]+)", header_text, re.IGNORECASE)
    if sz_match: meta['case_id'] = sz_match.group(1)
    
    # 3. ECLI - I added the dot [A-Z0-9:.]+ so it would take ending .14, which was cut until now
    ecli_match = re.search(r"(ECLI:[A-Z0-9:.]+)", header_text)
    if ecli_match: 
        meta['ecli'] = ecli_match.group(1)
    
    return meta


for doc in pipeline_data:
    # i look at the first 2000 chars (metadata block)
    header_snippet = doc["clean_pages"][0][:2000] if doc["clean_pages"] else ""
    doc["extracted_meta"] = extract_meta(header_snippet)
    
    print(f"  {doc['filename'][:30]}... -> {doc['extracted_meta']}")

  NS_SR_5Obdo_14_2023_00_dokumen... -> {'date_raw': '25.04.2024', 'case_id': '5Obdo/14/2023', 'ecli': 'ECLI:SK:NSSR:2024:7811201291.4'}
  KS_Banská_Bystrica_41Cob_3_202... -> {'date_raw': '10.09.2025', 'case_id': '41Cob/3/2025', 'ecli': 'ECLI:SK:KSBB:2025:6122355709.4'}
  KS_Košice_3Cob_219_2023_00_dok... -> {'date_raw': '31.07.2025', 'case_id': '3Cob/219/2023', 'ecli': 'ECLI:SK:KSKE:2025:6122327846.1'}
  KS_Bratislava_2Cob_140_2023_00... -> {'date_raw': '15.04.2025', 'case_id': '2Cob/140/2023', 'ecli': 'ECLI:SK:KSBA:2025:6120446490.2'}
  KS_Nitra_15Cob_70_2022_00_doku... -> {'date_raw': '25.07.2023', 'case_id': '15Cob/70/2022', 'ecli': 'ECLI:SK:KSNR:2023:4117226891.4'}


## 3.4 Deleting page numbers and main header separation

In [7]:

def remove_pagination(text: str) -> str:
    lines = text.split('\n')
    out = []
    for line in lines:
        s = line.strip()
        # Remove "Strana X", "- X -", but keeps for example "Spisová značka" etc.
        if re.match(r"^(?:strana\s+\d+(?:\s*z\s*\d+)?|\-?\s*\d+\s*\-?)$", s, re.IGNORECASE):    # Strana 3, Strana 1 z 10 and also "1", 
            continue
        out.append(line)
    return "\n".join(out)

def split_header_body(text: str):
    # Regex to find the start of the judgment body - there is the end of main header of metadata
    # Always in our case starts with "ROZSUDOK" or "UZNESENIE" in uppercase on a new line - it cuts exactly before this line
    split_pat = re.compile(r"\n\s*(?:ROZSUDOK|UZNESENIE|ROZSUDOK\s+V\s+MENE\s+SLOVENSKEJ\s+REPUBLIKY)\s*\n", re.IGNORECASE)
    match = split_pat.search(text)
    if match:
        return text[:match.start()].strip(), text[match.start():].strip()
    
    # Fallback to ECLI position - if there is no "rozsdok" or "uznesenie" then it cuts exactly after the ECLI 
    ecli_match = re.search(r"ECLI:[^\n]+\n", text)
    if ecli_match:
        return text[:ecli_match.end()].strip(), text[ecli_match.end():].strip()
        
    return text[:1000], text[1000:] # Last resort fallback



for doc in pipeline_data:
    # 1. Remove pagination from all pages
    no_page_nums = [remove_pagination(p) for p in doc["clean_pages"]]
    full_text = "\n".join(no_page_nums)
    
    # 2. Split Header vs Body
    h_text, b_text = split_header_body(full_text)
    
    doc["header_text_raw"] = h_text
    doc["body_text_raw"] = b_text
    
    # DEBUG SAVE
    with open(DEBUG_DIR / f"{doc['filename']}_03_header.txt", "w", encoding="utf-8") as f:
        f.write(h_text)
    with open(DEBUG_DIR / f"{doc['filename']}_03_body_raw.txt", "w", encoding="utf-8") as f:
        f.write(b_text)


## 3.5 Layout normalization ( connecting lines )

We need to create real sections 

In [8]:
# list of abbreviations. To know that it is not the end of sentence (just abbreviations)
ABBREVIATIONS = r"(?:ods\.|písm\.|č\.|čl\.|sp\.|zn\.|sp\.\s*zn\.|z\.\s*z\.|zb\.)"
# adding titles because there was problem with ending the line after them.
TITLES = r"(?:judr\.|mgr\.|ing\.|mudr\.|phdr\.|rndr\.|bc\.|doc\.|prof\.|akad\.|thlic\.|paeddr\.)"
# prepositions in slovak - if it is ath the end of line - it surely has to be merged with next line
PREPOSITIONS = r"(?:\s|^)(?:v|z|zo|k|ku|o|po|pri|pre|na|do)$"

def fix_spaced_words(text):
    # it finds: letter + space ( it has to be repeated at least 3x) + letter
    # Example: "S ú d" -> "Súd", but "a v" (spojky) leaves as it is.
    return re.sub(r'(?<!\S)((?:[a-zA-ZÁ-Žá-ž]\s){2,}[a-zA-ZÁ-Žá-ž])(?!\S)', 
                  lambda m: m.group(1).replace(" ", ""), text)

# this function finds the headers with spaces between each letters (O D O V O D N E N I E) - never merge with previous or next line
def is_spaced_header(text: str) -> bool:
    clean = text.rstrip(".:").strip()
    if len(clean) < 5: return False
    return re.match(r"^\s*(?:[a-zA-ZÁ-Žá-ž]\s+){3,}[a-zA-ZÁ-Žá-ž]\s*$", clean) is not None

# get line - should i paste it to what i have already in buffer? 
#               - if yes - add it to buffer
#               - if no - everything in buffer is now final section(paragraph)
def normalize_layout(text: str) -> str:
    lines = text.split('\n')
    merged = []
    buf = ""
    
    for line in lines:
        line = line.strip()
        if not line: # Empty line = hard paragraph break
            if buf != "": 
                merged.append(buf)
            buf = ""
            merged.append("")
            continue
        
        if buf == "":
            buf = line
            continue
            
        prev = buf
        should_join = False
        
        # A) JOIN RULES
        if prev.endswith("-"): # Hyphen - then we merge
            buf = prev[:-1] + line
            continue
        # section (buffer) cannot end with preposition at the end - merge it
        elif re.search(PREPOSITIONS, prev, re.IGNORECASE): 
            should_join = True
        # even when line ends with dot, it can be just abbreviations - join it
        elif re.search(rf"{ABBREVIATIONS}$", prev, re.IGNORECASE):
            should_join = True
        
        # B) STOP RULES
        elif is_spaced_header(line):    # headlines do not merge with previous line
            should_join = False
        elif prev.endswith(":"):    # double dot also do not merge
            should_join = False
        # if line ends with "SLOVENSKEJ REPUBLIKY", it means that it is header - do not merge !
        elif prev.strip().upper().endswith("SLOVENSKEJ REPUBLIKY"): should_join = False
        # ------------------------------------
        # List items (1., a)) - do not merge to previous line, but NOT dates (20. 1.)
        #       i deleteted '-', so it would not tear "Spisove znacky" ids (e.g. - 404)
        elif re.match(r"^(?:[a-z]\)|\d+\.(?!\s*\d)|•)\s", line):
            should_join = False
        # if prev line ends with dot or exclamation mark - do not merge
        elif line[0].isupper() and re.search(r"[.?!]$", prev):
            should_join = False
        
        # C) GENERAL - nothing from up do not apply - use very simple deduction
        else:
            if not re.search(r"[.?!]$", prev):
                should_join = True
            elif line[0].islower():
                should_join = True
            
        if should_join:
            buf += " " + line
        else:
            merged.append(buf); buf = line
            
    if buf: merged.append(buf)
    # return "\n".join(merged)
    return fix_spaced_words("\n".join(merged))



for doc in pipeline_data:
    # Runbing only on body text
    doc["body_text_layout"] = normalize_layout(doc["body_text_raw"])
    
    
    # DEBUG SAVE
    with open(DEBUG_DIR / f"{doc['filename']}_04_layout_fixed.txt", "w", encoding="utf-8") as f:
        f.write(doc["body_text_layout"])
        
    print(f"{doc['filename']}")
    print(f" Before (lines): {len(doc['body_text_raw'].splitlines())}")
    print(f" After (sections):   {len(doc['body_text_layout'].splitlines())}")
    print("-" * 45)


NS_SR_5Obdo_14_2023_00_dokument.pdf
 Before (lines): 925
 After (sections):   109
---------------------------------------------
KS_Banská_Bystrica_41Cob_3_2025_00_dokument.pdf
 Before (lines): 454
 After (sections):   79
---------------------------------------------
KS_Košice_3Cob_219_2023_00_dokument.pdf
 Before (lines): 366
 After (sections):   88
---------------------------------------------
KS_Bratislava_2Cob_140_2023_00_dokument.pdf
 Before (lines): 715
 After (sections):   73
---------------------------------------------
KS_Nitra_15Cob_70_2022_00_dokument.pdf
 Before (lines): 376
 After (sections):   48
---------------------------------------------


## 3.6 Advanced unicode normalization 
I ensure consistency here, so that machine will see only one type of hyphen and quotes

I replaced various types of dashes (long dashes –, —) with a standard hyphen -. I also replaced Slovak-style "curly quotes" („, “) with standard straight quotes (").

Different symbols for the same meaning could confuse search algorithms. Standardizing them ensures that when I search for a phrase, the system finds it regardless of which specific character was used in the original PDF. Could help in potential RAG system.

In [9]:
# Unifying dashes and quotes.
def advanced_unicode(text: str) -> str:
    # dashes
    t = text.replace('–', '-').replace('—', '-')
    # quotes
    t = t.replace('„', '"').replace('“', '"').replace('”', '"')
    return t


for doc in pipeline_data:
    doc["body_text_final"] = advanced_unicode(doc["body_text_layout"])
    
    # DEBUG SAVE for control
    with open(DEBUG_DIR / f"{doc['filename']}_05_unicode_final.txt", "w", encoding="utf-8") as f:
        f.write(doc["body_text_final"])

print("Final text is ready now.")

Final text is ready now.


## 3.7 Segmentation

In [10]:
def segment_body(text: str):
    segments = {"verdict": "", "reasoning": "", "instruction": ""}
    
    # Regexes ( it should cathc Odovodnenie with "o" and also with "ô" )
    re_reasoning = re.compile(r"(?:[oó]\s*d\s*[oóô]\s*v\s*o\s*d\s*n\s*e\s*n\s*i\s*e|odôvodnenie)", re.IGNORECASE)
    re_instruction = re.compile(r"(?:p\s*o\s*u\s*č\s*e\s*n\s*i\s*e|poučenie)", re.IGNORECASE)
    
    # 1. Find Reasoning start
    match_odovodnenie = re_reasoning.search(text)
    if match_odovodnenie is not None:
        print(match_odovodnenie)
        start_res = match_odovodnenie.start()
        # print(start_res)
        segments["verdict"] = text[:start_res].strip()      # everything before "Odovodnenie" is verdict
        rest = text[start_res:].strip()
        
        # 2. Find Instruction start ( poucenie - from the rest of text)
        match_poucenie = re_instruction.search(rest)
        if match_poucenie is not None:
            segments["reasoning"] = rest[:match_poucenie.start()].strip()
            segments["instruction"] = rest[match_poucenie.start():].strip() # poucenie
        else:
            segments["reasoning"] = rest
    else:
        # Fallback if Reasoning is missing
        segments["verdict"] = text
        
    return segments


for doc in pipeline_data:
    doc["segments"] = segment_body(doc["body_text_final"])
    
    # DEBUG SAVE (Formatted)
    debug_path = DEBUG_DIR / f"{doc['filename']}_06_segments.txt"
    with open(debug_path, "w", encoding="utf-8") as f:
        f.write(f"== HEADER ==\n{doc['header_text_raw']}\n\n")
        f.write(f"== VERDICT ==\n{doc['segments']['verdict']}\n\n")
        f.write(f"== REASONING ==\n{doc['segments']['reasoning']}\n\n")
        f.write(f"== INSTRUCTION ==\n{doc['segments']['instruction']}\n")



<re.Match object; span=(857, 868), match='odôvodnenie'>
<re.Match object; span=(4206, 4217), match='odôvodnenie'>
<re.Match object; span=(971, 982), match='odôvodnenie'>
<re.Match object; span=(936, 947), match='odôvodnenie'>
<re.Match object; span=(1182, 1193), match='odôvodnenie'>
